# Data Loading Tutorial

This tutorial covers how to load EEG data from various formats supported by NeuRodent.

## Overview

NeuRodent supports multiple data formats commonly used in rodent EEG research:

1. **SpikeInterface recordings** (`mode="si"`) - Any format supported by SpikeInterface (EDF, Intan, Open Ephys, NWB, etc.)
2. **MNE objects** (`mode="mne"`) - Any format supported by MNE-Python (FIF, EDF, BDF, etc.)
3. **Pre-created recordings** (`mode=None`) - Pass an existing `si.BaseRecording` directly
4. **Custom formats** - Via a custom `extract_func` callable or module path string

The `LongRecordingOrganizer` (LRO) class handles loading and organizing recordings from these formats.
The `AnimalOrganizer` class manages multiple sessions for a single animal using glob-style patterns.

## Setup

In [ ]:
import csv
from pathlib import Path
import logging

from datetime import datetime

import numpy as np

from neurodent import core
from neurodent.core.discovery import DiscoveredFile, FileDiscoverer

import spikeinterface.core as si


# Set up logging
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(message)s",
    level=logging.INFO
)
logger = logging.getLogger()

# Path to the included test data (bin/csv pairs)
# Adjust this path relative to your notebook location
DATA_ROOT = Path("../../tests/data/raw")

## 1. Loading Data with a Custom Reader

The repository includes small binary recordings in `tests/data/raw/`.
Each recording consists of a paired `.bin` data file and a `.csv` metadata
file.  We define a short reader function and pass it as `extract_func`.

For multi-file formats like this, wrap the paths in a `DiscoveredFile`
object so the LRO knows the files belong together.

In [ ]:
def read_bin_csv_pair(discovered_file, **kwargs):
    """Read paired ColMajor .bin + Meta .csv files into a recording."""
    bin_path = [p for p in discovered_file.paths if p.endswith(".bin")][0]
    csv_path = [p for p in discovered_file.paths if p.endswith(".csv")][0]

    with open(csv_path) as f:
        rows = list(csv.DictReader(f))

    n_channels = len(rows)
    sampling_rate = float(rows[0]["SampleRate"])
    channel_names = [row["Label"] for row in rows]
    data = np.fromfile(bin_path, dtype=np.float32).reshape(-1, n_channels)

    return si.NumpyRecording(
        traces_list=[data],
        sampling_frequency=sampling_rate,
        channel_ids=channel_names,
    )

In [ ]:
# Create a DiscoveredFile pointing to the A10 bin/csv pair
a10_dir = DATA_ROOT / "A10"
a10_discovered = DiscoveredFile(
    paths=(
        str(a10_dir / "Cage 2 A10-0_ColMajor.bin"),
        str(a10_dir / "Cage 2 A10-0_Meta.csv"),
    ),
)

# Load via LongRecordingOrganizer with mode="si"
lro = core.LongRecordingOrganizer(
    item=a10_discovered,
    mode="si",
    extract_func=read_bin_csv_pair,
)

print(f"Sampling frequency: {lro.meta.f_s} Hz")
print(f"Number of channels: {lro.meta.n_channels}")
print(f"Channel names: {lro.meta.channel_names}")

In [ ]:
# Access the underlying SpikeInterface recording
recording = lro.LongRecording

print(f"Recording type: {type(recording).__name__}")
print(f"Duration: {recording.get_total_duration():.1f} seconds")

### Using a File-Path String as ``extract_func``

Instead of defining the reader inline, you can point to a function in
any Python file using the ``"path/to/file.py:function_name"`` syntax.
The repository ships a reader at ``tests/data/readers.py``:

In [ ]:
# Equivalent to the inline reader above, but resolved from a file path.
# Adjust the path to readers.py relative to your working directory.
lro_from_path = core.LongRecordingOrganizer(
    item=a10_discovered,
    mode="si",
    extract_func="tests/data/readers.py:read_bin_csv_pair",
)

print(f"Sampling frequency: {lro_from_path.meta.f_s} Hz")
print(f"Number of channels: {lro_from_path.meta.n_channels}")

## 2. Pattern-Based File Discovery

When your data directory contains many recordings, `FileDiscoverer`
can find and pair files automatically using placeholder patterns
(e.g. `{animal}`, `{session}`, `{index}`).  For multi-file formats,
pass a **list** of patterns — one per file type — and files that share
the same placeholder values will be grouped together.

In [ ]:
# Discover all bin/csv pairs under tests/data/raw/
pattern = [
    str(DATA_ROOT / "{animal}" / "*_ColMajor.bin"),
    str(DATA_ROOT / "{animal}" / "*_Meta.csv"),
]

discoverer = FileDiscoverer(pattern)
discovered_files = discoverer.discover()

for f in discovered_files:
    print(f"Animal {f.metadata['animal']}: {[Path(p).name for p in f.paths]}")

In [ ]:
# Load a specific discovered file
f22_file = [f for f in discovered_files if f.metadata["animal"] == "F22"][0]

lro_f22 = core.LongRecordingOrganizer(
    item=f22_file,
    mode="si",
    extract_func=read_bin_csv_pair,
)

print(f"F22 sampling frequency: {lro_f22.meta.f_s} Hz")
print(f"F22 channels: {lro_f22.meta.channel_names}")
print(f"F22 duration: {lro_f22.LongRecording.get_total_duration():.1f} s")

## 3. Loading Standard Formats via SpikeInterface

For common single-file formats (EDF, Intan, NWB, etc.) you can pass a
SpikeInterface extractor name directly as `extract_func`.  NeuRodent
resolves the name from `spikeinterface.extractors`.

```python
# Example: Loading an EDF file (requires pyedflib)
lro_edf = core.LongRecordingOrganizer(
    item="/path/to/recording.edf",
    mode="si",
    extract_func="read_edf",
)

# Example: Loading Intan .rhd files
lro_intan = core.LongRecordingOrganizer(
    item="/path/to/recording.rhd",
    mode="si",
    extract_func="read_intan",
)
```

## 4. Loading MNE Objects

MNE-Python is widely used for MEG and EEG analysis.  NeuRodent can
load data via MNE with `mode="mne"` and an appropriate reader.

```python
import mne

lro_mne = core.LongRecordingOrganizer(
    item="/path/to/recording.fif",
    mode="mne",
    extract_func=mne.io.read_raw_fif,
    manual_datetimes=datetime(2023, 12, 13),
)
```

## 5. Inspecting Loaded Data

Once data is loaded, inspect its properties via the `meta` attribute
(a `RecordingMetadata` object):

In [ ]:
metadata = lro.meta

print(f"Recording metadata: {metadata}")
print(f"Sampling frequency: {metadata.f_s} Hz")
print(f"Number of channels: {metadata.n_channels}")
print(f"Channel names: {metadata.channel_names}")
print(f"Units: {metadata.V_units}")
print(f"Duration: {lro.file_durations} seconds")

## 6. Loading Other Formats

NeuRodent supports many more data formats.  The sections below show
common patterns — replace the paths with your own data.

### Neuroscope / Neuralynx

```python
import spikeinterface.extractors as se

recording_neuroscope = se.read_neuroscope("/path/to/data.dat")
lro = core.LongRecordingOrganizer(item=None, mode=None, recording=recording_neuroscope)
```

### Open Ephys

```python
recording_oe = se.read_openephys("/path/to/openephys/folder")
lro = core.LongRecordingOrganizer(item=None, mode=None, recording=recording_oe)
```

### NWB

```python
recording_nwb = se.read_nwb("/path/to/file.nwb")
lro = core.LongRecordingOrganizer(item=None, mode=None, recording=recording_nwb)
```

## 7. Working with Multiple Recordings

NeuRodent can concatenate multiple single-file recordings from the
same animal (e.g. different sessions or days):

```python
lro_multi = core.LongRecordingOrganizer(
    item=["/path/to/session1.edf", "/path/to/session2.edf"],
    mode="si",
    extract_func="read_edf",
)
```

## 8. Advanced: In-Memory Custom Data

For custom formats or testing, create a SpikeInterface `NumpyRecording`
and pass it directly to `LongRecordingOrganizer` with `mode=None`:

In [ ]:
# Create a recording from a numpy array
num_channels = 8
sampling_frequency = 1000  # Hz
duration = 10  # seconds
num_samples = int(sampling_frequency * duration)

data = np.random.randn(num_samples, num_channels).astype(np.float32)

recording_custom = si.NumpyRecording(
    traces_list=[data],
    sampling_frequency=sampling_frequency,
)

channel_ids = [f"CH{i:02d}" for i in range(num_channels)]
recording_custom = recording_custom.rename_channels(new_channel_ids=channel_ids)

lro_custom = core.LongRecordingOrganizer(
    item=None,
    mode=None,
    recording=recording_custom,
)

print("Custom recording created successfully!")
print(f"Sampling frequency: {lro_custom.meta.f_s}")
print(f"Number of channels: {lro_custom.meta.n_channels}")
print(f"Channel names: {lro_custom.meta.channel_names}")

## Summary

In this tutorial, you learned:

1. How to load paired binary/CSV data with a custom `extract_func` and `DiscoveredFile`
2. How to use `FileDiscoverer` to automatically find and pair recordings
3. How to load standard formats (EDF, Intan, NWB) via SpikeInterface or MNE
4. How to inspect loaded data properties via the `meta` attribute
5. How to create in-memory recordings from NumPy arrays

## Next Steps

- **[Basic Usage Tutorial](basic_usage.ipynb)**: Complete workflow from loading to visualization
- **[Windowed Analysis Tutorial](../tutorials/windowed_analysis.ipynb)**: Extract features from loaded data
- **[Spike Analysis Tutorial](../tutorials/spike_analysis.ipynb)**: Work with spike-sorted data